# Method Comparison — Options Pricing Engine

Five pricing engines built during Milestones 1–5:
Black-Scholes closed-form, CRR binomial tree, Crank-Nicolson PDE,
Monte Carlo, and Heston (Gil-Pelaez inversion).

This notebook validates each engine, measures empirical convergence,
compares runtime-vs-accuracy tradeoffs, cross-checks American put pricing,
and previews the Heston implied volatility smile as the transition
to Project 2 (`volatility-surface-lab`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from src.models import black_scholes as bs
from src.models.gbm import GBMModel
from src.models.heston import HestonModel
from src.engines.binomial_tree import price_option as tree_price
from src.engines.pde_solver import PDESolver
from src.engines.monte_carlo import MonteCarloEngine
from src.payoffs.european import EuropeanCallPayoff, EuropeanPutPayoff
from src.utils.implied_vol import implied_vol

from src.comparison.helpers import (
    sweep_tree, sweep_pde, sweep_mc,
    sweep_american_put,
    plot_convergence, plot_pareto, plot_american_put, plot_heston_smile,
)

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
np.random.seed(42)

## Section 1 — Sanity Check

All five engines pricing the same European call ($S=K=100$, $T=1$, $r=0.05$, $\sigma=0.20$). Closed-form Black-Scholes is the reference. For the Heston row, parameters are chosen to collapse the model to Black-Scholes (variance pinned at $\sigma^2$, vol-of-vol $\approx 0$).

Monte Carlo agreement is reported as a $z$-score against the closed-form price; $|z| < 3$ is passing (results within 3 standard errors).

In [ ]:
bs_price = bs.call_price(S, K, T, r, sigma)

tree_p = tree_price(S, K, T, r, sigma, N=500, option_type="call", exercise="european")

gbm = GBMModel(S0=S, r=r, sigma=sigma, T=T)
call_payoff = EuropeanCallPayoff(K=K, T=T)
pde = PDESolver(gbm, call_payoff, n_space=400, n_time=5000)
pde_p = pde.price(S0=S, T=T)

mc = MonteCarloEngine(gbm, call_payoff, r=r, T=T)
mc_res = mc.price(n_paths=100_000, antithetic=True, control_variate=True)
mc_p, mc_se = mc_res["price"], mc_res["std_error"]
mc_z = (mc_p - bs_price) / mc_se

heston_bs = HestonModel(S0=S, r=r, v0=sigma**2, kappa=2.0, theta=sigma**2,
                        sigma_v=1e-4, rho=0.0)
h_p = heston_bs.price_call(K=K, T=T)

df = pd.DataFrame([
    {"Method": "BS closed-form",     "Price": bs_price, "|Error|": 0.0,               "Z-score": "—"},
    {"Method": "CRR tree (N=500)",   "Price": tree_p,   "|Error|": abs(tree_p-bs_price), "Z-score": "—"},
    {"Method": "CN PDE (400×5000)",  "Price": pde_p,    "|Error|": abs(pde_p-bs_price),  "Z-score": "—"},
    {"Method": "MC (100k, anti+CV)", "Price": mc_p,     "|Error|": abs(mc_p-bs_price),   "Z-score": f"{mc_z:+.2f}"},
    {"Method": "Heston (BS limit)",  "Price": h_p,      "|Error|": abs(h_p-bs_price),    "Z-score": "—"},
])
df.style.format({"Price": "{:.6f}", "|Error|": "{:.2e}"})

## Section 2 — Convergence Rates

Empirical convergence for each numerical method against theoretical rates: CRR tree $O(1/N)$, Crank-Nicolson $O(\Delta x^2)$, Monte Carlo standard error $O(1/\sqrt{n})$. MC sweep uses vanilla (no variance reduction) to measure the clean rate — antithetic/control-variate shift the intercept, not the slope.

In [ ]:
def sweep_tree(S, K, T, r, sigma, bs_price, N_grid):
    errors, times = [], []
    for N in N_grid:
        t0 = time.perf_counter()
        p = tree_price(S, K, T, r, sigma, N=N, option_type="call", exercise="european")
        times.append(time.perf_counter() - t0)
        errors.append(abs(p - bs_price))
    return np.array(N_grid), np.array(errors), np.array(times)

def sweep_pde(gbm, call_payoff, S, T, bs_price, nspace_grid, n_time=5000):
    errors, times = [], []
    for n_space in nspace_grid:
        pde = PDESolver(gbm, call_payoff, n_space=n_space, n_time=n_time)
        t0 = time.perf_counter()
        p = pde.price(S0=S, T=T)
        times.append(time.perf_counter() - t0)
        errors.append(abs(p - bs_price))
    return np.array(nspace_grid), np.array(errors), np.array(times)

def sweep_mc(gbm, call_payoff, r, T, bs_price, npaths_grid, seed=42,
             antithetic=False, control_variate=False):
    np.random.seed(seed)
    errors, times, ses = [], [], []
    for n_paths in npaths_grid:
        mc = MonteCarloEngine(gbm, call_payoff, r=r, T=T)
        t0 = time.perf_counter()
        res = mc.price(n_paths=n_paths, antithetic=antithetic, control_variate=control_variate)
        times.append(time.perf_counter() - t0)
        errors.append(abs(res["price"] - bs_price))
        ses.append(res["std_error"])
    return np.array(npaths_grid), np.array(errors), np.array(times), np.array(ses)

def plot_convergence(tree_data, pde_data, mc_data):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    specs = [
        (axes[0], tree_data, "CRR tree",    "N (steps)", "|error|", -1.0),
        (axes[1], pde_data,  "CN PDE",      "n_space",   "|error|", -2.0),
        (axes[2], mc_data,   "Monte Carlo", "n_paths",   "SE",      -0.5),
    ]
    for ax, (xs, ys), title, xlab, ylab, theory_slope in specs:
        ax.loglog(xs, ys, "o-", label="empirical")
        slope, _ = np.polyfit(np.log(xs), np.log(ys), 1)
        ref = ys[0] * (xs / xs[0]) ** theory_slope
        ax.loglog(xs, ref, "--", alpha=0.6, label=f"theory slope {theory_slope:+.1f}")
        ax.set_xlabel(xlab); ax.set_ylabel(ylab)
        ax.set_title(f"{title}  (fitted slope {slope:+.2f})")
        ax.grid(True, which="both", alpha=0.3); ax.legend()
    fig.suptitle("Section 2: Convergence rates vs. theory")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

N_grid      = [50, 100, 200, 500, 1000, 2000, 5000]
nspace_grid = [50, 100, 200, 400, 800]
npaths_grid = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]

tree_N, tree_err, tree_t = sweep_tree(S, K, T, r, sigma, bs_price, N_grid)
pde_ns, pde_err, pde_t   = sweep_pde(gbm, call_payoff, S, T, bs_price, nspace_grid)
mc_np, mc_err_v, mc_t_v, mc_se_v = sweep_mc(gbm, call_payoff, r, T, bs_price, npaths_grid)

for name, xs, ys, theory in [("Tree", tree_N, tree_err, -1),
                              ("PDE",  pde_ns, pde_err, -2),
                              ("MC",   mc_np,  mc_se_v, -0.5)]:
    slope, _ = np.polyfit(np.log(xs), np.log(ys), 1)
    print(f"{name:5s} empirical slope: {slope:+.3f}  (theory: {theory})")

fig_conv = plot_convergence((tree_N, tree_err), (pde_ns, pde_err), (mc_np, mc_se_v))
plt.show()

## Section 3 — Runtime vs. Accuracy

Convergence rate tells you how error shrinks with grid refinement; the Pareto view tells you which method actually wins at a given accuracy target. For 1D vanilla payoffs, the tree dominates low-to-mid accuracy, Crank-Nicolson takes over at high precision, and Monte Carlo — even with variance reduction — is dominated across the board. That's the right answer: MC's comparative advantage is high-dimensional or path-dependent problems.

In [ ]:
def plot_pareto(tree_data, pde_data, mc_vanilla, mc_vr, heston_point=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    series = [
        ("CRR tree",     tree_data[2],   tree_data[1],   "o"),
        ("CN PDE",       pde_data[2],    pde_data[1],    "s"),
        ("MC (vanilla)", mc_vanilla[2],  mc_vanilla[1],  "^"),
        ("MC (anti+CV)", mc_vr[2],       mc_vr[1],       "v"),
    ]
    for label, times, errs, marker in series:
        ax.loglog(times, errs, marker=marker, linestyle="-", label=label, alpha=0.85)
    if heston_point is not None:
        t, e = heston_point
        ax.loglog([t], [e], marker="*", markersize=15, linestyle="none",
                  label="Heston (BS limit)")
    ax.set_xlabel("Runtime (seconds)")
    ax.set_ylabel("|error|")
    ax.set_title("Section 3: Runtime vs. accuracy Pareto frontier")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return fig

# MC with variance reduction (fresh sweep, same grid)
mc_np2, mc_err_vr, mc_t_vr, _ = sweep_mc(
    gbm, call_payoff, r, T, bs_price, npaths_grid,
    antithetic=True, control_variate=True,
)

# Heston BS-limit as a single reference point
t0 = time.perf_counter()
h_ref = HestonModel(S0=S, r=r, v0=sigma**2, kappa=2.0, theta=sigma**2,
                    sigma_v=1e-4, rho=0.0)
h_price = h_ref.price_call(K=K, T=T)
h_time = time.perf_counter() - t0
h_err = abs(h_price - bs_price)
print(f"Heston BS-limit: runtime {h_time:.3f}s   error {h_err:.2e}")

fig_pareto = plot_pareto(
    (tree_N, tree_err, tree_t),
    (pde_ns, pde_err,  pde_t),
    (mc_np,  mc_err_v, mc_t_v),
    (mc_np2, mc_err_vr, mc_t_vr),
    heston_point=(h_time, h_err),
)
plt.show()

## Section 4 — American Put

No closed form under Black-Scholes, so we validate two ways: (1) cross-check tree vs. Crank-Nicolson with early-exercise across strikes; (2) structural — the early exercise premium should be strictly positive and monotone in moneyness. Right panel shows the free-boundary $S^*(t)$ from the PDE solver.

In [ ]:
def sweep_american_put(S, K_grid, T, r, sigma, N_tree=2000, n_space=400, n_time=5000):
    euro_bs, amer_tree, amer_pde = [], [], []
    for Ki in K_grid:
        euro_bs.append(bs.put_price(S, Ki, T, r, sigma))
        amer_tree.append(tree_price(S, Ki, T, r, sigma, N=N_tree,
                                    option_type="put", exercise="american"))
        gbm_k = GBMModel(S0=S, r=r, sigma=sigma, T=T)
        put_payoff = EuropeanPutPayoff(K=Ki, T=T)
        pde_k = PDESolver(gbm_k, put_payoff, n_space=n_space, n_time=n_time)
        amer_pde.append(pde_k.price(S0=S, T=T, american=True))
    return {"K": np.array(K_grid), "euro_bs": np.array(euro_bs),
            "amer_tree": np.array(amer_tree), "amer_pde": np.array(amer_pde)}

def plot_american_put(results, S, T, r, sigma, n_space=400, n_time=5000):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    Kg = results["K"]
    ax = axes[0]
    ax.plot(Kg, results["euro_bs"],   "k--", label="European put (BS)")
    ax.plot(Kg, results["amer_tree"], "o-",  label="American put (tree)")
    ax.plot(Kg, results["amer_pde"],  "s-",  label="American put (PDE)")
    ax.set_xlabel("Strike K"); ax.set_ylabel("Put price")
    ax.set_title(f"American vs European put  (S={S}, T={T})")
    ax.grid(True, alpha=0.3); ax.legend()
    ax2 = ax.twinx()
    premium = results["amer_tree"] - results["euro_bs"]
    ax2.plot(Kg, premium, "r:", marker="d", label="Early exercise premium")
    ax2.set_ylabel("Early exercise premium", color="r")
    ax2.tick_params(axis="y", labelcolor="r")
    ax2.legend(loc="lower right")

    K_atm = S
    gbm_atm = GBMModel(S0=S, r=r, sigma=sigma, T=T)
    put_atm = EuropeanPutPayoff(K=K_atm, T=T)
    pde_atm = PDESolver(gbm_atm, put_atm, n_space=n_space, n_time=n_time)
    _ = pde_atm.price(S0=S, T=T, american=True)

    ax = axes[1]
    ax.plot(pde_atm.boundary_times, pde_atm.exercise_boundary, "b-", lw=1.2)
    ax.axhline(K_atm, color="k", linestyle="--", alpha=0.5, label=f"Strike K={K_atm}")
    ax.set_xlabel("Time t"); ax.set_ylabel("Exercise boundary S*(t)")
    ax.set_title(f"Early exercise boundary  (American put, K={K_atm})")
    ax.grid(True, alpha=0.3); ax.legend()
    ax.invert_xaxis()
    fig.tight_layout()
    return fig

K_grid = [90, 95, 100, 105, 110, 115, 120]
amer = sweep_american_put(S, K_grid, T=1.0, r=r, sigma=sigma)

amer_df = pd.DataFrame({
    "K":              amer["K"],
    "Euro (BS)":      amer["euro_bs"],
    "Amer (tree)":    amer["amer_tree"],
    "Amer (PDE)":     amer["amer_pde"],
    "tree - PDE":     amer["amer_tree"] - amer["amer_pde"],
    "Exercise prem":  amer["amer_tree"] - amer["euro_bs"],
})
display(amer_df.style.format({
    "K": "{:.0f}",
    "Euro (BS)": "{:.6f}", "Amer (tree)": "{:.6f}", "Amer (PDE)": "{:.6f}",
    "tree - PDE": "{:.2e}", "Exercise prem": "{:.6f}",
}))

fig_amer = plot_american_put(amer, S=S, T=1.0, r=r, sigma=sigma)
plt.show()

## Section 5 — Heston Implied Volatility Smile

Under Black-Scholes, implied vol is a constant; under Heston, it's a smile — stochastic vol produces fatter tails than lognormal, and the market prices that in. Correlation $\rho$ controls the skew: $\rho<0$ gives the classical equity shape where OTM puts are expensive (leverage effect); $\rho>0$ mirrors it (commodities-style). This is the transition point to Project 2 (`volatility-surface-lab`), where calibrating the surface becomes the whole game.

**Feller condition note:** parameters here violate $2\kappa\theta \geq \sigma_v^2$ — that's the empirically relevant regime for equity vol markets. The Gil-Pelaez pricer is unaffected (works in Fourier space); the constraint matters for Monte Carlo schemes and for calibration stability in Project 2.

In [ ]:
def plot_heston_smile(S, T, r, K_grid, rho_list):
    fig, ax = plt.subplots(figsize=(9, 6))
    v0, kappa, theta, sigma_v = 0.04, 2.0, 0.04, 0.5
    for rho in rho_list:
        heston = HestonModel(S0=S, r=r, v0=v0, kappa=kappa, theta=theta,
                             sigma_v=sigma_v, rho=rho)
        ivs = []
        for Ki in K_grid:
            price = heston.price_call(K=Ki, T=T)
            iv = implied_vol(price, S=S, K=Ki, T=T, r=r, option_type="call")
            ivs.append(iv)
        ax.plot(K_grid, ivs, "o-", label=f"ρ = {rho:+.2f}", alpha=0.85)
    ax.axhline(np.sqrt(v0), color="k", linestyle=":", alpha=0.5,
               label=f"BS flat vol (√v₀ = {np.sqrt(v0):.2f})")
    ax.axvline(S, color="gray", linestyle="--", alpha=0.4, label="ATM")
    ax.set_xlabel("Strike K"); ax.set_ylabel("Black-Scholes implied volatility")
    ax.set_title(f"Section 5: Heston implied vol smile  (T={T}, transition to Project 2)")
    ax.grid(True, alpha=0.3); ax.legend()
    fig.tight_layout()
    return fig

K_smile = np.linspace(70, 130, 25)
fig_smile = plot_heston_smile(S=S, T=0.5, r=r, K_grid=K_smile,
                              rho_list=[-0.7, 0.0, +0.7])
plt.show()

In [ ]:
os.makedirs('../assets', exist_ok=True)
fig_conv.savefig('../assets/convergence.png', dpi=120, bbox_inches='tight')
fig_pareto.savefig('../assets/pareto.png', dpi=120, bbox_inches='tight')
fig_smile.savefig('../assets/heston_smile.png', dpi=120, bbox_inches='tight')
print("Saved: convergence.png, pareto.png, heston_smile.png")